> **This notebook is designed to run on a Databricks cluster. It reads Snowflake-managed Iceberg tables via the Horizon Iceberg REST Catalog (IRC).**

# Databricks — Horizon IRC Multi-Engine Read

**Quickstart: Snowflake + Iceberg Interoperability** (Module 4)

Reads Snowflake-managed Iceberg tables from Databricks via Horizon REST Catalog.

## Required Cluster Configuration (DBR 17.3)

| Setting | Value |
|---|---|
| Databricks Runtime | **17.3** (Spark 4.0 — required for Iceberg V3 VARIANT) |
| Photon | **Disabled** (conflicts with Iceberg runtime at this version) |
| Access mode | **No isolation shared** (Unity Catalog mode blocks direct REST catalog connections) |

**Cluster Libraries (install via Compute → Libraries → Maven):**
- `org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1`
- `org.apache.iceberg:iceberg-aws-bundle:1.10.1`
- `net.snowflake:spark-snowflake_2.13:3.1.6` *(transitively includes `snowflake-jdbc:3.24.2` — do **not** install JDBC separately)*

> **Why the Snowflake Connector for Spark?**
> Tables with masking or row-access policies (Path B) cannot vend storage credentials.
> The `SnowflakeFallbackCatalog` detects this and routes the query through Snowflake
> compute so policies are enforced server-side. Path A tables still use vended credentials.

---

### Configuration Notes

1. **Scala 2.13, not 2.12** — DBR 17.3 ships Scala 2.13. Using `spark-snowflake_2.12` causes `NoClassDefFoundError: scala/collection/GenTraversableOnce`.
2. **Do NOT install `snowflake-jdbc` as a separate cluster library** — `spark-snowflake_2.13:3.1.6` transitively pulls `snowflake-jdbc:3.24.2`. Installing `3.24.0` explicitly causes a version conflict.
3. **Key pair auth for Spark Connector** — PAT works for the Iceberg REST catalog credential but does **not** work for the Spark Connector JDBC fallback (Path B). Use `pem_private_key` (RSA base64, no PEM headers) for `spark.snowflake.*` config.
4. **`.cast("string")` on VARIANT columns** — Iceberg V3 VARIANT reads as `variant` type in Spark 4.0. `get_json_object` requires a string input, so cast explicitly.
5. **`TIMESTAMP_NTZ(6)` in table DDL** — Default Snowflake precision maps to `timestamp_ns` in Iceberg metadata. Iceberg Spark runtime 1.10.1 cannot read `timestamp_ns`. Explicit `::TIMESTAMP_NTZ(6)` forces microsecond precision (Iceberg `timestamptz`/`timestamp` type).
6. **`spark.jars.packages` in notebook code is a no-op on Databricks** — DBR pre-initializes SparkSession. All JARs must be installed as cluster-level Maven libraries. The `spark.jars.packages` lines in this notebook are kept for documentation only.

---

Fill in `ACCOUNT_IDENTIFIER`, `PAT_TOKEN`, `RSA_PRIVATE_KEY`, and `REGION` below. See [DATABRICKS_SETUP.md](../DATABRICKS_SETUP.md) for detailed cluster configuration and troubleshooting.

In [ ]:
ACCOUNT_IDENTIFIER = "<YOUR_ACCOUNT_IDENTIFIER>"  # e.g. "ORGNAME-ACCTNAME" (run: SELECT CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME())
PAT_TOKEN          = "<YOUR_PAT_TOKEN>"
ROLE_NAME          = "DEMO_DBX_READER"
DATABASE_NAME      = "ICEBERG_DEMO"
SCHEMA_NAME        = "PUBLIC"
REGION             = "us-east-1"
ICEBERG_VERSION    = "1.10.1"

SF_URL             = f"{ACCOUNT_IDENTIFIER}.snowflakecomputing.com"
SF_USER            = "<YOUR_SNOWFLAKE_USER>"
SF_WAREHOUSE       = "DEMO_WH"
SPARK_CONNECTOR_VER = "3.1.6"
RSA_PRIVATE_KEY    = "<YOUR_RSA_PRIVATE_KEY_BASE64>"

## Initialize SparkSession with Horizon IRC

In [ ]:
from pyspark.sql import SparkSession

CATALOG_URI = f"https://{SF_URL}/polaris/api/catalog"
SCOPE = f"session:role:{ROLE_NAME}"
CAT   = DATABASE_NAME

spark = (
    SparkSession.builder
    .appName("Quickstart_Horizon_IRC")
    .config("spark.jars.packages",
        f"org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:{ICEBERG_VERSION},"
        f"org.apache.iceberg:iceberg-aws-bundle:{ICEBERG_VERSION},"
        f"net.snowflake:spark-snowflake_2.13:{SPARK_CONNECTOR_VER}")
    .config("spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.defaultCatalog", CAT)
    .config(f"spark.sql.catalog.{CAT}",
        "org.apache.spark.sql.snowflake.catalog.SnowflakeFallbackCatalog")
    .config(f"spark.sql.catalog.{CAT}.catalog-impl",
        "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{CAT}.type", "rest")
    .config(f"spark.sql.catalog.{CAT}.uri", CATALOG_URI)
    .config(f"spark.sql.catalog.{CAT}.warehouse", CAT)
    .config(f"spark.sql.catalog.{CAT}.credential", PAT_TOKEN)
    .config(f"spark.sql.catalog.{CAT}.scope", SCOPE)
    .config(f"spark.sql.catalog.{CAT}.client.region", REGION)
    .config(f"spark.sql.catalog.{CAT}.header.X-Iceberg-Access-Delegation", "vended-credentials")
    .config(f"spark.sql.catalog.{CAT}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.snowflake.sfURL", SF_URL)
    .config("spark.snowflake.sfUser", SF_USER)
    .config("spark.snowflake.pem_private_key", RSA_PRIVATE_KEY)
    .config("spark.snowflake.sfDatabase", DATABASE_NAME)
    .config("spark.snowflake.sfSchema", SCHEMA_NAME)
    .config("spark.snowflake.sfRole", ROLE_NAME)
    .config("spark.snowflake.sfWarehouse", SF_WAREHOUSE)
    .config("spark.sql.iceberg.vectorization.enabled", "false")
    .getOrCreate()
)

print(f"SparkSession initialized.")
print(f"Catalog URI : {CATALOG_URI}")
print(f"Role scope  : {SCOPE}")
print(f"SF Warehouse: {SF_WAREHOUSE} (used for policy-routed Path B compute)")

In [ ]:
spark.sql(f"SHOW NAMESPACES IN {CAT}").show(truncate=False)
spark.sql(f"SHOW TABLES IN {CAT}.{SCHEMA_NAME}").show(truncate=False)

## PATH A — yellow_trips (no policy, vended credentials, zero SF compute)


**Vended credentials:** Snowflake issues short-lived AWS STS tokens so Databricks can read Parquet files directly from Snowflake-managed S3 storage — no Snowflake warehouse needed.
Horizon vends temporary STS credentials. Databricks reads Parquet files
directly from Snowflake-managed storage. No Snowflake warehouse compute consumed.

**Watch the Snowflake credit meter — it will not move during this read.**

In [ ]:
import time

yellow_df = spark.table(f"{CAT}.{SCHEMA_NAME}.YELLOW_TRIPS")

print("Schema:")
yellow_df.printSchema()
print(f"\nRow count: {yellow_df.count():,}")

start = time.time()
result = spark.sql(f"""
    SELECT
        PULocationID,
        COUNT(*)                        AS trip_count,
        ROUND(SUM(fare_amount),  2)     AS total_fare,
        ROUND(AVG(tip_amount),   2)     AS avg_tip,
        ROUND(AVG(trip_distance),2)     AS avg_distance_miles
    FROM {CAT}.{SCHEMA_NAME}.YELLOW_TRIPS
    GROUP BY PULocationID
    ORDER BY trip_count DESC
    LIMIT 15
""")
result.show(truncate=False)
print(f"Query time: {time.time() - start:.1f}s")

## Nanosecond Timestamp — Iceberg V3 vs Spark Compatibility

Snowflake supports `TIMESTAMP_NTZ(9)` (nanosecond) natively under Iceberg V3.
`green_trips_nano` is identical to `green_trips` but with nanosecond timestamps.
Snowflake reads it fine. **Spark 1.10.x cannot** — it rejects `timestamp_ns` at schema level.

This is why the main tables use `TIMESTAMP_NTZ(6)` (microsecond) — an intentional
compatibility choice, not a Snowflake limitation.

In [ ]:
try:
    nano_df = spark.table(f"{CAT}.{SCHEMA_NAME}.GREEN_TRIPS_NANO")
    nano_df.show(5)
except Exception as e:
    print("Expected failure reading GREEN_TRIPS_NANO (nanosecond timestamps):")
    print(f"  {type(e).__name__}: {str(e)[:300]}")
    print()
    print("Spark 1.10.x cannot read Iceberg timestamp_ns.")
    print("The fix: use TIMESTAMP_NTZ(6) in the table DDL — microsecond precision.")

In [ ]:
green_micro_df = spark.table(f"{CAT}.{SCHEMA_NAME}.GREEN_TRIPS")
print(f"GREEN_TRIPS (microsecond TIMESTAMP_NTZ(6)) — reads successfully:")
print(f"Row count: {green_micro_df.count():,}")
green_micro_df.select("VendorID", "lpep_pickup_datetime", "fare_amount").show(5, truncate=False)
print("Same data, same Iceberg V3 format. Only difference: timestamp precision.")

## PATH B — green_trips (masking + RLS active, routed through SF compute)


**Routed through SF compute:** Snowflake intercepts the request, executes the query on a Snowflake warehouse, applies masking/RLS policies, and returns only the filtered/masked results to Databricks.
Snowflake detects the active policies. The Databricks query is routed through Snowflake compute.
`fare_amount` and `tip_amount` will return **-1.0** (masked). RLS filters rows to Manhattan only.

**Now the Snowflake credit meter will move.**

In [ ]:
green_df = spark.table(f"{CAT}.{SCHEMA_NAME}.GREEN_TRIPS")
print(f"Row count (RLS-filtered, Manhattan only): {green_df.count():,}")

spark.sql(f"""
    SELECT
        PULocationID,
        fare_amount,
        tip_amount,
        total_amount,
        trip_distance
    FROM {CAT}.{SCHEMA_NAME}.GREEN_TRIPS
    LIMIT 10
""").show(truncate=False)

print("Note: fare_amount = -1.0 and tip_amount = -1.0 confirm masking policy is active.")
print("Note: Row count filtered by RLS -- DEMO_DBX_READER sees Manhattan pickups only.")

## Iceberg V3 VARIANT — nyc_weather_ssv2 (read from Databricks)


Data streamed into this table by `SSV2WeatherIngest.java` via Snowpipe Streaming V2.
Iceberg V3 VARIANT columns are natively supported in Spark 4.0 via the open-source `variant_get` function (PySpark + Spark SQL).
Extract fields with inline type coercion — no cast-to-string, no JSON parsing, no schema pre-definition.

**Requires DBR 17.3 + Spark 4.0. VARIANT is not readable on earlier DBR versions.**

In [ ]:
from pyspark.sql.functions import variant_get, col

weather_df = spark.table(f"{CAT}.{SCHEMA_NAME}.NYC_WEATHER_SSV2")

print("Schema (VARIANT column reads as native 'variant' type in Spark 4.0):")
weather_df.printSchema()

print("Raw VARIANT data (vertical view):")
weather_df.select("LOCATION", "LATITUDE", "LONGITUDE", "WEATHER_DATA").show(3, truncate=50, vertical=True)

weather_df.select(
    col("location"),
    variant_get("weather_data", "$.timezone",                  "string").alias("timezone"),
    variant_get("weather_data", "$.elevation",                 "float").alias("elevation_m"),
    variant_get("weather_data", "$.hourly_units.precipitation", "string").alias("precip_unit")
).show(truncate=False)

## Summary

| Table | Path | SF Compute | Masking | RLS |
|---|---|---|---|---|
| `yellow_trips` | A | Zero (vended creds, direct storage) | None | None |
| `green_trips` | B | Consumed (SF compute routing) | -1.0 on financial | Manhattan only |
| `nyc_weather_ssv2` | A | Zero | None | None |

Same Horizon IRC endpoint. Same Databricks cluster. Governance enforced at the catalog level.

In [ ]:
spark.stop()
print("SparkSession stopped.")